In [1]:
import pandas as pd 

In [2]:
col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

Loading data from raw data file 

In [5]:
train_data = pd.read_csv("../data/KDDTrain+.txt", header =None, names=col_names)
test_data = pd.read_csv("../data/KDDTest+.txt", header =None, names=col_names)

print("Train data shape:", train_data.shape)
print("Train data shape:", train_data.shape)

Train data shape: (125973, 43)
Train data shape: (125973, 43)


Creating Binary Label       

In [6]:
train_data["binary_label"] = train_data["label"].apply(lambda x: "normal" if x == "normal" else "attack")
test_data["binary_label"] = test_data["label"].apply(lambda x: "normal" if x == "normal" else "attack")

print(train_data["binary_label"].value_counts())

binary_label
normal    67343
attack    58630
Name: count, dtype: int64


In [8]:
print(train_data.head(5))

   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_diff_srv_rate  \
0               0       0    0  ...                    0.03   
1               0       0    0  ...                    0.60   
2               0       0    0  ...                    0.05   
3               0       0    0  ...                    0.00   
4               0       0    0  ...                    0.00   

   dst_host_same_src_port_rate  dst_host_srv_diff_host_rate  \
0                         0.17                         0.00   
1                         0.88                         0.00   

Encoding 

In [9]:
from sklearn.preprocessing import OneHotEncoder

categorical_colmns = ["protocol_type", "service", "flag"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output= False)

encoder.fit(train_data[categorical_colmns])


,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [10]:
train_data_encoded = encoder.transform(train_data[categorical_colmns]) #transforming categorical comlumns of train data into encoded version
test_data_encoded = encoder.transform(test_data[categorical_colmns])

encoded_colmn_names = encoder.get_feature_names_out(categorical_colmns)

train_data_encoded_df = pd.DataFrame(train_data_encoded, columns=encoded_colmn_names, index= train_data.index)
test_data_encoded_df = pd.DataFrame(test_data_encoded, columns=encoded_colmn_names, index = test_data.index)

print("Train Data fram shape",train_data_encoded_df.shape)
print("Test Data fram shape",test_data_encoded_df.shape)

Train Data fram shape (125973, 84)
Test Data fram shape (22544, 84)


## One-Hot Encoding

Three columns (`protocol_type`, `service`, `flag`) are text, not numbers — models need numbers.

One-hot encoding turns each category into its own 0/1 column (e.g., `service_http`, `flag_SF`), instead of assigning arbitrary numbers that would imply a false ranking between categories.

**Key rule: fit only on train, transform both train and test** — this prevents data leakage, where information from test data would accidentally influence how the model is built.

`handle_unknown="ignore"` — safely handles categories that appear in test but never appeared in train (we know this happens with `service`, since NSL-KDD's test set has a slightly different set of services than train).

In [11]:
from sklearn.preprocessing import StandardScaler

exclude_colmns = categorical_colmns + ["label", "binary_label", "difficulty"]
numeric_colmns = [col for col in train_data.columns if col not in exclude_colmns]

scaler = StandardScaler()
scaler.fit(train_data[numeric_colmns])

train_data_scaled = scaler.transform(train_data[numeric_colmns])
test_data_scaled = scaler.transform(test_data[numeric_colmns])

train_data_scaled_df = pd.DataFrame(train_data_scaled, columns=numeric_colmns, index=train_data.index)
test_data_scaled_df = pd.DataFrame(test_data_scaled, columns=numeric_colmns, index=test_data.index)

print("Train scaled data shape", train_data_scaled_df.shape)
print("Test scaled data shape",test_data_scaled_df.shape)

Train scaled data shape (125973, 38)
Test scaled data shape (22544, 38)


## Scaling Numeric Features

The 38 numeric columns have very different ranges (e.g. `duration` can be huge, `land` is just 0/1). Some models (like Logistic Regression) are sensitive to this — a large-range feature could dominate just because of its size, not its actual importance.

`StandardScaler` transforms each column to have mean 0 and standard deviation 1, putting every feature on a comparable scale.

Same rule as encoding: fit only on train, transform both.

Merging data

In [12]:
train_data_final = pd.concat([train_data_scaled_df, train_data_encoded_df, train_data["binary_label"]], axis=1)
test_data_final = pd.concat([test_data_scaled_df, test_data_encoded_df, test_data["binary_label"]], axis=1)

print("Train final shape:", train_data_final.shape)
print("Test final shape:", test_data_final.shape)

train_data_final.to_csv("../data/processed_train_REBUILT.csv", index=False)
test_data_final.to_csv("../data/processed_test_REBUILT.csv", index=False)

print("Saved to all data files for comparison")

Train final shape: (125973, 123)
Test final shape: (22544, 123)
Saved to all data files for comparison


In [13]:
original = pd.read_csv("../data/processed_train.csv")
rebuilt = pd.read_csv("../data/processed_train_REBUILT.csv")

print("Same shape:", original.shape == rebuilt.shape)
print("Identical values:", original.equals(rebuilt))

Same shape: True
Identical values: True
